# Transformers

In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import seaborn as snss
import pandas as pd
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['axes.grid'] = True
plt.rcParams['axes.axisbelow'] = True
plt.rcParams['figure.dpi'] = 150

In [2]:
# Check available hardware
import torch
if torch.cuda.is_available():
    print(f'GPU is available, using GPU {torch.cuda.get_device_name(0)}')
    device = torch.device('cuda')
elif torch.mps.is_available():
    print(f'Apple Metal Performance Shaders framework is available, using {torch.device("mps")}')
    device = torch.device('mps')
else:
    print(f'Hardware acceleration not available, using CPU')
    device = torch.device('cpu')

GPU is available, using GPU Tesla T4


## Implementing scaled dot product attention

In [67]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import math



seq = torch.rand(1, 3, 4)
print('Input sequence')
print(seq)
print()

W_q = torch.rand(2, 4)
W_k = torch.rand(2, 4)
W_v = torch.rand(2, 4)

# torch version
print('Output is')
F.scaled_dot_product_attention(
    seq @ W_q.T, seq @ W_k.T, seq @ W_v.T
)

Input sequence
tensor([[[0.6621, 0.7961, 0.4889, 0.3158],
         [0.2462, 0.9663, 0.6324, 0.6379],
         [0.0082, 0.7920, 0.8047, 0.3106]]])

Output is


tensor([[[0.9021, 0.6354],
         [0.9046, 0.6364],
         [0.8990, 0.6346]]])

In [68]:
def scaled_dot_product_attention(query,
                                 key,
                                 value,
                                 W_q,
                                 W_k,
                                 W_v):
    """ Query, Key, Value are batched sequences of vectors of size [B, N, F]
        W_q, W_k, W_v are matrices that map QKV to an embedding space of dim E
    """
    # Projection weights
    q = query @ W_q.T #[B, N, E]
    k = key @ W_k.T #[B, N, E]
    v = value @ W_v.T #[B, N, E]

    # Compute unnormalized attention score
    a_unnorm = torch.einsum('bie,bje->bij', q, k) / math.sqrt(q.shape[-1])

    # Normalize using softmax
    a_norm = F.softmax(a_unnorm, dim=-1)

    print('Aggregating with attention weights')
    print(a_norm)
    print()

    # Perform weighted sum over values
    output = torch.einsum('bij,bje->bie', a_norm, v)
    return output

result = scaled_dot_product_attention(seq, seq, seq, W_q, W_k, W_v)
print('Output is')
print(result)

Aggregating with attention weights
tensor([[[0.3905, 0.3429, 0.2665],
         [0.3966, 0.3457, 0.2577],
         [0.3810, 0.3412, 0.2778]]])

Output is
tensor([[[0.9021, 0.6354],
         [0.9046, 0.6364],
         [0.8990, 0.6346]]])


## Implementing multi-head self-attention

In [69]:
mha = nn.MultiheadAttention(embed_dim=4, num_heads=2, batch_first=True)

for param in mha.named_parameters():
    print(param)
    print(param[1].shape)

('in_proj_weight', Parameter containing:
tensor([[-0.4700, -0.4005,  0.2386,  0.1024],
        [ 0.3446, -0.6116, -0.2633,  0.0790],
        [-0.2171, -0.5724,  0.0204,  0.0676],
        [-0.1846, -0.3596,  0.4572,  0.5150],
        [ 0.1375,  0.0256, -0.1758,  0.2387],
        [-0.1675, -0.0626, -0.1750, -0.1868],
        [ 0.2821, -0.4234,  0.2062,  0.2275],
        [-0.2435, -0.0702, -0.4317,  0.0434],
        [-0.1261,  0.1734,  0.1922, -0.0034],
        [ 0.3301, -0.5145, -0.0508, -0.1397],
        [ 0.0550, -0.1357, -0.4329,  0.2735],
        [-0.5081, -0.3203,  0.0770, -0.6020]], requires_grad=True))
torch.Size([12, 4])
('in_proj_bias', Parameter containing:
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], requires_grad=True))
torch.Size([12])
('out_proj.weight', Parameter containing:
tensor([[ 0.3176,  0.3575,  0.0026, -0.2813],
        [ 0.0495, -0.1265, -0.1163, -0.3656],
        [ 0.1900,  0.3078, -0.3246,  0.2197],
        [-0.0471, -0.4062, -0.3231,  0.4383]], requ

In [72]:
outputs, attn_weights = mha(seq, seq, seq)

print('Attention weights')
print(attn_weights)

print('Outputs')
print(outputs)

Attention weights
tensor([[[0.3265, 0.3333, 0.3403],
         [0.3274, 0.3360, 0.3366],
         [0.3296, 0.3371, 0.3334]]], grad_fn=<MeanBackward1>)
Outputs
tensor([[[ 0.0965,  0.3267, -0.1413, -0.0276],
         [ 0.0965,  0.3266, -0.1413, -0.0278],
         [ 0.0967,  0.3267, -0.1415, -0.0282]]], grad_fn=<TransposeBackward0>)


In [84]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, input_dim=4, embed_dim=4, num_heads=2):
        super().__init__()

        self.W_q = nn.Parameter(torch.rand(input_dim, num_heads, embed_dim // num_heads))
        self.W_k = nn.Parameter(torch.rand(input_dim, num_heads, embed_dim // num_heads))
        self.W_v = nn.Parameter(torch.rand(input_dim, num_heads, embed_dim // num_heads))

        self.W_o = nn.Parameter(torch.rand(embed_dim, embed_dim))

    def forward(self, seq):
        # Seq has shape [B, N, I]
        print(seq.shape, self.W_q.shape)
        q = torch.einsum('bni,ihe->bnhe', seq, self.W_q) # [B, N, H, E]
        k = torch.einsum('bni,ihe->bnhe', seq, self.W_k) # [B, N, H, E]
        v = torch.einsum('bni,ihe->bnhe', seq, self.W_v) # [B, N, H, E]
        print(q.shape, k.shape, v.shape)

        a_unnorm = torch.einsum('bihe,bjhe->bhij', q, k) / math.sqrt(q.shape[-1])
        print(a_unnorm.shape)

        a_norm = F.softmax(a_unnorm, dim=-1)
        print(a_norm.shape)

        output_heads = torch.einsum('bhij,bjhe->bihe', a_norm, v)
        print(output_heads.shape)
        
        # Concatenate heads
        B, N, H, E = output_heads.shape
        output = output_heads.reshape([B, N, H*E])
        print(output.shape)

        # Apply output projection matrix
        output = output @ self.W_o.T
        return output

mhsa = MultiHeadSelfAttention()
mhsa(seq)

torch.Size([1, 3, 4]) torch.Size([4, 2, 2])
torch.Size([1, 3, 2, 2]) torch.Size([1, 3, 2, 2]) torch.Size([1, 3, 2, 2])
torch.Size([1, 2, 3, 3])
torch.Size([1, 2, 3, 3])
torch.Size([1, 3, 2, 2])
torch.Size([1, 3, 4])


tensor([[[2.0097, 3.0089, 1.4320, 2.3492],
         [2.0177, 3.0209, 1.4390, 2.3572],
         [2.0050, 2.9972, 1.4268, 2.3386]]], grad_fn=<UnsafeViewBackward0>)

In [73]:
W_q = mha.in_proj_weight[0:4, :]
W_k = mha.in_proj_weight[4:8, :]
W_v = mha.in_proj_weight[8:12, :]



## PyTorch Transformer

In [3]:
import torch
import torch.nn as nn

transformer = nn.Transformer(nhead=4, num_encoder_layers=2, batch_first=True)
transformer

Transformer(
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (linear1): Linear(in_features=512, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=512, bias=True)
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
    (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (decoder): TransformerDecoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerDecoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, o